<a href="https://colab.research.google.com/github/67160330/week7/blob/main/pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import os
import sqlite3
import logging
from dataclasses import dataclass, field
from datetime import datetime
from typing import List, Tuple
import pandas as pd

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

@dataclass
class PipelineConfig:
    input_path: str = "Python_Data_Pipeline_Lab_Dataset (1).xlsx"
    db_path: str = "retail_dw.db"
    quarantine_csv: str = "quarantine.csv"
    log_csv: str = "pipeline_run_log.csv"
    batch_list: List[str] = field(default_factory=lambda: [
        "orders_batch_1",
        "orders_batch_1",  # Test Idempotency
        "orders_batch_2",
        "orders_batch_3"
    ])
    error_mode: str = "quarantine"

PAYMENT_METHOD_MAP = {
    'cash': 'Cash',
    'bank transfer': 'Bank Transfer',
    'promptpay': 'PromptPay',
    'credit card': 'Credit Card'
}

SALES_CHANNEL_MAP = {
    'store': 'Store',
    'online': 'Online',
    'marketplace': 'Marketplace',
    'e-commerce': 'Online'
}

def init_database(db_path: str):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("PRAGMA foreign_keys = ON;")

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS dim_customer (
            customer_key INTEGER PRIMARY KEY AUTOINCREMENT,
            customer_id TEXT UNIQUE NOT NULL,
            customer_name TEXT,
            province TEXT,
            segment TEXT
        );
    """)
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS dim_product (
            product_key INTEGER PRIMARY KEY AUTOINCREMENT,
            product_id TEXT UNIQUE NOT NULL,
            product_name TEXT,
            category TEXT
        );
    """)
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS dim_date (
            date_key INTEGER PRIMARY KEY,
            full_date TEXT UNIQUE NOT NULL,
            day INTEGER,
            month INTEGER,
            quarter INTEGER,
            year INTEGER
        );
    """)
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS fact_sales (
            order_id TEXT PRIMARY KEY,
            date_key INTEGER NOT NULL,
            customer_key INTEGER NOT NULL,
            product_key INTEGER NOT NULL,
            quantity INTEGER NOT NULL,
            unit_price REAL NOT NULL,
            discount_pct REAL NOT NULL,
            gross_amount REAL NOT NULL,
            net_amount REAL NOT NULL,
            payment_method TEXT,
            sales_channel TEXT,
            updated_at TEXT,
            FOREIGN KEY (date_key) REFERENCES dim_date(date_key),
            FOREIGN KEY (customer_key) REFERENCES dim_customer(customer_key),
            FOREIGN KEY (product_key) REFERENCES dim_product(product_key)
        );
    """)
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS quarantine (
            order_id TEXT, order_datetime TEXT, customer_id TEXT,
            product_id TEXT, quantity TEXT, unit_price TEXT,
            discount_pct TEXT, payment_method TEXT, sales_channel TEXT,
            updated_at TEXT, source_batch TEXT, reason_code TEXT, quarantined_at TEXT
        );
    """)
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS pipeline_run_log (
            run_id INTEGER PRIMARY KEY AUTOINCREMENT,
            batch TEXT NOT NULL,
            started_at TEXT NOT NULL,
            ended_at TEXT NOT NULL,
            rows_read INTEGER NOT NULL,
            rows_valid INTEGER NOT NULL,
            rows_rejected INTEGER NOT NULL,
            rows_duplicated INTEGER NOT NULL,
            rows_loaded INTEGER NOT NULL,
            net_sales_added REAL NOT NULL,
            status TEXT NOT NULL
        );
    """)
    conn.commit()
    conn.close()

def load_dimensions(config: PipelineConfig):
    xls = pd.ExcelFile(config.input_path)
    df_cust = pd.read_excel(xls, 'customers')
    df_prod = pd.read_excel(xls, 'products')

    conn = sqlite3.connect(config.db_path)
    cursor = conn.cursor()
    for _, row in df_cust.iterrows():
        cursor.execute("""
            INSERT INTO dim_customer (customer_id, customer_name, province, segment)
            VALUES (?, ?, ?, ?)
            ON CONFLICT(customer_id) DO UPDATE SET
                customer_name=excluded.customer_name,
                province=excluded.province,
                segment=excluded.segment;
        """, (str(row['customer_id']).strip(), row['customer_name'], row['province'], row['segment']))

    for _, row in df_prod.iterrows():
        cursor.execute("""
            INSERT INTO dim_product (product_id, product_name, category)
            VALUES (?, ?, ?)
            ON CONFLICT(product_id) DO UPDATE SET
                product_name=excluded.product_name,
                category=excluded.category;
        """, (str(row['product_id']).strip(), row['product_name'], row['category']))
    conn.commit()
    conn.close()

def extract_batch(config: PipelineConfig, batch_name: str) -> pd.DataFrame:
    xls = pd.ExcelFile(config.input_path)
    return pd.read_excel(xls, batch_name)

def transform_and_validate(df: pd.DataFrame, batch_name: str, config: PipelineConfig) -> Tuple[pd.DataFrame, pd.DataFrame, int]:
    conn = sqlite3.connect(config.db_path)
    valid_cust = set(pd.read_sql("SELECT customer_id FROM dim_customer", conn)['customer_id'])
    valid_prod = set(pd.read_sql("SELECT product_id FROM dim_product", conn)['product_id'])
    conn.close()

    df_raw = df.copy()
    df['parsed_order_datetime'] = pd.to_datetime(df['order_datetime'], errors='coerce')
    df['parsed_updated_at'] = pd.to_datetime(df['updated_at'], errors='coerce')
    df['parsed_quantity'] = pd.to_numeric(df['quantity'], errors='coerce')
    df['parsed_unit_price'] = pd.to_numeric(df['unit_price'], errors='coerce')
    df['parsed_discount_pct'] = pd.to_numeric(df['discount_pct'], errors='coerce')

    df['clean_customer_id'] = df['customer_id'].astype(str).str.strip()
    df['clean_product_id'] = df['product_id'].astype(str).str.strip()
    df['clean_payment'] = df['payment_method'].astype(str).str.strip().str.lower().map(PAYMENT_METHOD_MAP)
    df['clean_channel'] = df['sales_channel'].astype(str).str.strip().str.lower().map(SALES_CHANNEL_MAP)

    reasons = []
    for idx, row in df.iterrows():
        errs = []
        if pd.isna(row['parsed_order_datetime']): errs.append("INVALID_ORDER_DATETIME")
        if pd.isna(row['parsed_updated_at']): errs.append("INVALID_UPDATED_AT")
        if pd.isna(row['parsed_quantity']) or row['parsed_quantity'] <= 0 or (row['parsed_quantity'] % 1 != 0) or row['parsed_quantity'] > 20:
            errs.append("INVALID_QUANTITY")
        if pd.isna(row['parsed_unit_price']) or row['parsed_unit_price'] <= 0:
            errs.append("INVALID_UNIT_PRICE")
        if pd.isna(row['parsed_discount_pct']) or not (0 <= row['parsed_discount_pct'] <= 100):
            errs.append("INVALID_DISCOUNT_PCT")
        if pd.isna(row['clean_customer_id']) or row['clean_customer_id'] not in valid_cust:
            errs.append("MISSING_CUSTOMER_REF")
        if pd.isna(row['clean_product_id']) or row['clean_product_id'] not in valid_prod:
            errs.append("MISSING_PRODUCT_REF")
        if pd.isna(row['clean_payment']): errs.append("INVALID_PAYMENT_METHOD")
        if pd.isna(row['clean_channel']): errs.append("INVALID_SALES_CHANNEL")
        reasons.append("|".join(errs) if errs else "VALID")

    df['reason_code'] = reasons
    is_valid = df['reason_code'] == "VALID"
    clean_df = df[is_valid].copy()

    quarantine_df = df_raw[~is_valid].copy()
    quarantine_df['reason_code'] = df[~is_valid]['reason_code']
    quarantine_df['source_batch'] = batch_name
    quarantine_df['quarantined_at'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

    clean_df['order_datetime'] = clean_df['parsed_order_datetime']
    clean_df['updated_at'] = clean_df['parsed_updated_at']
    clean_df['quantity'] = clean_df['parsed_quantity'].astype(int)
    clean_df['unit_price'] = clean_df['parsed_unit_price']
    clean_df['discount_pct'] = clean_df['parsed_discount_pct']
    clean_df['customer_id'] = clean_df['clean_customer_id']
    clean_df['product_id'] = clean_df['clean_product_id']
    clean_df['payment_method'] = clean_df['clean_payment']
    clean_df['sales_channel'] = clean_df['clean_channel']
    clean_df['gross_amount'] = clean_df['quantity'] * clean_df['unit_price']
    clean_df['net_amount'] = clean_df['gross_amount'] * (1 - clean_df['discount_pct'] / 100.0)

    before_dedup = len(clean_df)
    clean_df = clean_df.sort_values('updated_at').groupby('order_id').last().reset_index()
    duplicated_count = before_dedup - len(clean_df)

    return clean_df, quarantine_df, duplicated_count

def load_fact_and_date(clean_df: pd.DataFrame, config: PipelineConfig) -> Tuple[int, float]:
    conn = sqlite3.connect(config.db_path)
    cursor = conn.cursor()
    cust_map = pd.read_sql("SELECT customer_id, customer_key FROM dim_customer", conn).set_index('customer_id')['customer_key'].to_dict()
    prod_map = pd.read_sql("SELECT product_id, product_key FROM dim_product", conn).set_index('product_id')['product_key'].to_dict()
    existing_orders = pd.read_sql("SELECT order_id, updated_at FROM fact_sales", conn).set_index('order_id')['updated_at'].to_dict()

    rows_loaded = 0
    net_sales_added = 0.0

    for _, row in clean_df.iterrows():
        dt = row['order_datetime']
        date_key = int(dt.strftime('%Y%m%d'))
        cursor.execute("""
            INSERT INTO dim_date (date_key, full_date, day, month, quarter, year)
            VALUES (?, ?, ?, ?, ?, ?)
            ON CONFLICT(date_key) DO NOTHING;
        """, (date_key, dt.strftime('%Y-%m-%d'), dt.day, dt.month, (dt.month - 1) // 3 + 1, dt.year))

        cust_key = cust_map[row['customer_id']]
        prod_key = prod_map[row['product_id']]
        updated_str = row['updated_at'].strftime('%Y-%m-%d %H:%M:%S')
        order_id_str = str(row['order_id'])

        is_new = order_id_str not in existing_orders
        is_updated = False
        if not is_new and updated_str > existing_orders[order_id_str]:
            is_updated = True

        cursor.execute("""
            INSERT INTO fact_sales (
                order_id, date_key, customer_key, product_key, quantity, unit_price,
                discount_pct, gross_amount, net_amount, payment_method, sales_channel, updated_at
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            ON CONFLICT(order_id) DO UPDATE SET
                date_key=excluded.date_key, customer_key=excluded.customer_key,
                product_key=excluded.product_key, quantity=excluded.quantity,
                unit_price=excluded.unit_price, discount_pct=excluded.discount_pct,
                gross_amount=excluded.gross_amount, net_amount=excluded.net_amount,
                payment_method=excluded.payment_method, sales_channel=excluded.sales_channel,
                updated_at=excluded.updated_at
            WHERE excluded.updated_at >= fact_sales.updated_at;
        """, (
            order_id_str, date_key, cust_key, prod_key,
            int(row['quantity']), float(row['unit_price']), float(row['discount_pct']),
            float(row['gross_amount']), float(row['net_amount']),
            str(row['payment_method']), str(row['sales_channel']), updated_str
        ))

        if is_new or is_updated:
            rows_loaded += 1
            net_sales_added += row['net_amount']

    conn.commit()
    conn.close()
    return rows_loaded, net_sales_added

def run_pipeline(config: PipelineConfig, batch_name: str):
    started_at = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    rows_read = rows_valid = rows_rejected = rows_duplicated = rows_loaded = 0
    net_sales_added = 0.0
    status = "SUCCESS"

    try:
        raw_df = extract_batch(config, batch_name)
        rows_read = len(raw_df)

        clean_df, quarantine_df, rows_duplicated = transform_and_validate(raw_df, batch_name, config)
        rows_rejected = len(quarantine_df)
        rows_valid = len(clean_df) + rows_duplicated

        if not quarantine_df.empty:
            q_export = quarantine_df[[
                'order_id', 'order_datetime', 'customer_id', 'product_id',
                'quantity', 'unit_price', 'discount_pct', 'payment_method',
                'sales_channel', 'updated_at', 'source_batch', 'reason_code', 'quarantined_at'
            ]]
            conn = sqlite3.connect(config.db_path)
            q_export.to_sql('quarantine', conn, if_exists='append', index=False)
            conn.close()

            hdr = not os.path.exists(config.quarantine_csv)
            q_export.to_csv(config.quarantine_csv, mode='a', header=hdr, index=False)

        rows_loaded, net_sales_added = load_fact_and_date(clean_df, config)
        ended_at = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

    except Exception as e:
        ended_at = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        status = f"FAILED: {str(e)}"
        logging.error(f"Pipeline failed for {batch_name}: {e}")

    conn = sqlite3.connect(config.db_path)
    cursor = conn.cursor()
    cursor.execute("""
        INSERT INTO pipeline_run_log (
            batch, started_at, ended_at, rows_read, rows_valid, rows_rejected,
            rows_duplicated, rows_loaded, net_sales_added, status
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?);
    """, (batch_name, started_at, ended_at, rows_read, rows_valid, rows_rejected, rows_duplicated, rows_loaded, net_sales_added, status))
    conn.commit()

    log_df = pd.read_sql("SELECT * FROM pipeline_run_log", conn)
    log_df.to_csv(config.log_csv, index=False)
    conn.close()

if __name__ == "__main__":
    cfg = PipelineConfig()
    for f in [cfg.db_path, cfg.quarantine_csv, cfg.log_csv]:
        if os.path.exists(f):
            os.remove(f)

    init_database(cfg.db_path)
    load_dimensions(cfg)

    for b in cfg.batch_list:
        run_pipeline(cfg, b)